# ArchiLector · Entrenamiento del clasificador de páginas

Segunda IA propia (Documento de Diseño 4.6): clasifica cada página como `texto`, `formulario_tabla`,
`portada_indice` o `blanco`. En modo archivo organiza el PDF; en modo lectura permite saltar las páginas
en blanco y avisar «esta página tiene una tabla».

Datos: unas 150 fotos propias por categoría, corregidas en el panel web (**Documentos → página → corregir**)
y exportadas con **Etiquetar → Exportar clasificador (.zip)**. Opcionalmente se completan con el conjunto
público RVL-CDIP (Harley et al., 2015). Meta: F1 ≥ 0,85, comparado con la regla de palabras clave.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

subido = files.upload()  # clasificador.zip
CARPETA = Path("datos_clasificador")
zipfile.ZipFile(next(iter(subido))).extractall(CARPETA)

CLASES = ["texto", "formulario_tabla", "portada_indice", "blanco"]  # mismo orden que en la Pi
COLUMNA = "categoria"
NOMBRE = "clasificador"

import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

def dividir_por_grupo(df, semilla=42):
    """70 % entrenamiento, 15 % validación y 15 % prueba, separando por grupo.

    El grupo es el libro. Con pocos libros (3 en la meta del proyecto) no se
    puede dejar un libro entero para cada conjunto, así que se agrupan bloques
    de 25 capturas consecutivas del mismo libro: páginas casi iguales nunca
    quedan repartidas entre entrenamiento y prueba.
    """
    df = df.copy()
    if df["libro"].nunique() >= 6:
        df["grupo"] = df["libro"]
    else:
        df["grupo"] = df["libro"] + "_" + (df["id"] // 25).astype(str)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=semilla)
    i_ent, i_resto = next(gss.split(df, groups=df["grupo"]))
    resto = df.iloc[i_resto]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=semilla)
    i_val, i_pru = next(gss2.split(resto, groups=resto["grupo"]))
    return df.iloc[i_ent], resto.iloc[i_val], resto.iloc[i_pru]

meta = pd.read_csv(CARPETA / "meta.csv")
meta["id"] = meta["archivo"].str.extract(r"__(\d+)\.jpg$").astype(int)
print(meta[COLUMNA].value_counts())

## Opcional: completar con RVL-CDIP

RVL-CDIP tiene 16 tipos de documentos escaneados. Se toman unas pocas imágenes por tipo, solo para
**entrenamiento** (la prueba se hace con fotos propias), y se agrupan en nuestras categorías.
No trae páginas en blanco: esas salen solo de las fotos propias.

In [ ]:
USAR_RVL = False
POR_TIPO = 60

if USAR_RVL:
    !pip -q install datasets
    from datasets import load_dataset
    # 0 carta, 1 formulario, 2 correo, 3 manuscrito, 4 publicidad, 5 informe científico,
    # 6 publicación científica, 7 especificación, 8 carpeta, 9 noticia, 10 presupuesto,
    # 11 factura, 12 presentación, 13 cuestionario, 14 currículum, 15 memorando
    MAPA = {0: "texto", 2: "texto", 5: "texto", 6: "texto", 9: "texto", 15: "texto",
            1: "formulario_tabla", 7: "formulario_tabla", 10: "formulario_tabla",
            11: "formulario_tabla", 13: "formulario_tabla", 8: "portada_indice", 12: "portada_indice"}
    cuenta, filas = {k: 0 for k in MAPA}, []
    for i, ej in enumerate(load_dataset("aharley/rvl_cdip", split="train", streaming=True)):
        k = ej["label"]
        if k in MAPA and cuenta[k] < POR_TIPO:
            destino = CARPETA / "rvl" / f"{k}_{cuenta[k]}.jpg"
            destino.parent.mkdir(parents=True, exist_ok=True)
            ej["image"].convert("RGB").save(destino)
            filas.append({"archivo": f"rvl/{destino.name}", COLUMNA: MAPA[k], "libro": "rvl", "id": 100000 + i})
            cuenta[k] += 1
        if all(v >= POR_TIPO for v in cuenta.values()):
            break
    rvl = pd.DataFrame(filas)
    print("RVL-CDIP:", rvl[COLUMNA].value_counts().to_dict())

In [ ]:
propias = meta.copy()
ent, val, pru = dividir_por_grupo(propias)
if USAR_RVL:
    ent = pd.concat([ent, rvl], ignore_index=True)  # RVL solo en entrenamiento
for nombre, d in (("entrenamiento", ent), ("validación", val), ("prueba", pru)):
    print(f"{nombre:>13}: {len(d):4d} ·", dict(d[COLUMNA].value_counts()))

In [ ]:
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

TAM = 224
LOTE = 32

def cargar(ruta):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    return tf.image.resize(img, (TAM, TAM))  # píxeles 0..255: el modelo se reescala solo

aumentar = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.02),          # rotación leve
    tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),  # cambios de luz
    tf.keras.layers.RandomContrast(0.15),
], name="aumentos")

def nitidez_aleatoria(img):
    # Desenfoca o enfoca un poco (tolerancia a la nitidez de la cámara)
    k = tf.constant([[1, 2, 1], [2, 4, 2], [1, 2, 1]], tf.float32) / 16.0
    k = tf.tile(k[:, :, None, None], [1, 1, 3, 1])
    suave = tf.nn.depthwise_conv2d(img[None], k, [1, 1, 1, 1], "SAME")[0]
    a = tf.random.uniform([], -0.6, 0.6)
    return tf.clip_by_value(img + a * (img - suave), 0, 255)

def conjunto(df, entrenar):
    rutas = [str(CARPETA / a) for a in df["archivo"]]
    etiquetas = [CLASES.index(c) for c in df[COLUMNA]]
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if entrenar:
        ds = ds.shuffle(len(rutas), seed=42)
    ds = ds.map(lambda r, y: (cargar(r), y), num_parallel_calls=tf.data.AUTOTUNE)
    if entrenar:
        ds = ds.map(lambda x, y: (nitidez_aleatoria(aumentar(x, training=True)), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(LOTE).prefetch(tf.data.AUTOTUNE)

ds_ent, ds_val, ds_pru = conjunto(ent, True), conjunto(val, False), conjunto(pru, False)

pesos = compute_class_weight("balanced", classes=np.arange(len(CLASES)),
                             y=[CLASES.index(c) for c in ent[COLUMNA]])
pesos = dict(enumerate(pesos))

base = tf.keras.applications.MobileNetV2(input_shape=(TAM, TAM, 3), include_top=False,
                                         weights="imagenet", pooling="avg")
base.trainable = False
entrada = tf.keras.Input((TAM, TAM, 3))
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(entrada)  # preprocesado de MobileNetV2 dentro del modelo
x = base(x, training=False)
x = tf.keras.layers.Dropout(0.3)(x)
salida = tf.keras.layers.Dense(len(CLASES), activation="softmax")(x)
modelo = tf.keras.Model(entrada, salida)

# Etapa 1: solo la capa nueva
modelo.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", ["accuracy"])
parar = tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)
h1 = modelo.fit(ds_ent, validation_data=ds_val, epochs=15, class_weight=pesos, callbacks=[parar])

# Etapa 2: también las últimas capas de la red (transferencia de aprendizaje)
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
modelo.compile(tf.keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", ["accuracy"])
h2 = modelo.fit(ds_ent, validation_data=ds_val, epochs=15, class_weight=pesos, callbacks=[parar])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, recall_score

y_real = np.array([CLASES.index(c) for c in pru[COLUMNA]])
y_pred = np.argmax(modelo.predict(ds_pru), axis=1)

print(classification_report(y_real, y_pred, labels=range(len(CLASES)), target_names=CLASES, digits=3, zero_division=0))
cm = confusion_matrix(y_real, y_pred, labels=range(len(CLASES)))
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(xticks_rotation=30, cmap="Blues")
plt.title("Matriz de confusión · conjunto de prueba")
plt.tight_layout(); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

## Comparación con la regla de palabras clave

La columna `prediccion_regla` es lo que dijo la Pi cuando no tenía modelo (regla de tinta, líneas y palabras clave).

In [ ]:
y_regla = np.array([CLASES.index(c) if c in CLASES else 0 for c in pru["prediccion_regla"]])
print(classification_report(y_real, y_regla, labels=range(len(CLASES)), target_names=CLASES, digits=3, zero_division=0))
f1_ia, f1_regla = f1_score(y_real, y_pred, average="macro"), f1_score(y_real, y_regla, average="macro")
print(f"F1 IA {f1_ia:.3f} · F1 regla {f1_regla:.3f} · meta ≥ 0,85: {'CUMPLE' if f1_ia >= 0.85 else 'NO CUMPLE'}")

## Exportar

Copie `clasificador.tflite` y `clasificador.txt` a `pi/modelos/` en la Raspberry Pi.

In [ ]:
# TensorFlow Lite cuantizado a 8 bits (pesos y activaciones), con entrada y
# salida en float32: la Pi le pasa los píxeles 0..255 tal cual.
def representativo():
    for x, _ in ds_ent.unbatch().take(200):
        yield [tf.cast(x[None], tf.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representativo
tflite = conv.convert()
open(f"{NOMBRE}.tflite", "wb").write(tflite)
open(f"{NOMBRE}.txt", "w").write("\n".join(CLASES))  # el orden de las clases viaja con el modelo
print(f"{NOMBRE}.tflite: {len(tflite) / 1e6:.1f} MB")

# Comprobación: el modelo cuantizado acierta lo mismo que el original
interp = tf.lite.Interpreter(model_content=tflite)
interp.allocate_tensors()
ent_d, sal_d = interp.get_input_details()[0], interp.get_output_details()[0]
aciertos = 0
for x, y in ds_pru.unbatch():
    interp.set_tensor(ent_d["index"], tf.cast(x[None], ent_d["dtype"]))
    interp.invoke()
    aciertos += int(np.argmax(interp.get_tensor(sal_d["index"])[0]) == int(y))
print(f"Exactitud TFLite en prueba: {aciertos / len(pru):.3f}")

from google.colab import files
for f in (f"{NOMBRE}.tflite", f"{NOMBRE}.txt", "matriz_confusion.png"):
    files.download(f)